# 03 - Fine-tuning de BETO para ABSA

Fine-tuneamos `dccuchile/bert-base-spanish-wwm-uncased` usando el formato de
auxiliary sentence (Sun et al., 2019):

```
[CLS] reseña [SEP] aspecto [SEP]  -> {pos, neg, neu}
```

In [ ]:
import sys
from pathlib import Path


def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("No se encontró la raíz del repositorio")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from sklearn.model_selection import GroupShuffleSplit, train_test_split

from src.data.loader import load_reviews
from src.data.builder import build_best_available_dataset
from src.evaluation.metrics import evaluate_predictions
from src.transformers_models.beto import BETOAspectClassifier


## 1. Preparar dataset

In [ ]:
df = load_reviews(REPO_ROOT / "data/sample/reviews_sample.csv")
texts, aspects, labels, label_source = build_best_available_dataset(df)
print(f"Ejemplos: {len(texts)} | fuente de etiquetas: {label_source}")
if label_source != "manual":
    print("Prueba funcional con pseudo-etiquetas; no reportar estas métricas como resultado final.")


## 2. Split entrenamiento / test

In [ ]:
group_column = None
if "product_id" in df.columns and df["product_id"].nunique() > 1:
    group_column = "product_id"
elif "review_id" in df.columns and df["review_id"].nunique() > 1:
    group_column = "review_id"

if group_column:
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(splitter.split(df, groups=df[group_column]))
    train_df = df.iloc[train_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)
else:
    train_df, test_df = train_test_split(
        df,
        test_size=0.2,
        random_state=42,
        stratify=df["rating"] if df["rating"].value_counts().min() >= 2 else None,
    )

t_tr, a_tr, y_tr, train_source = build_best_available_dataset(train_df)
t_te, a_te, y_te, test_source = build_best_available_dataset(test_df)
print(f"Train: {len(t_tr)} | Test: {len(t_te)} | etiquetas: {train_source}/{test_source}")


## 3. Cargar e instanciar BETO

**Nota:** la primera ejecución descarga ~440MB.

In [ ]:
clf = BETOAspectClassifier()
print('Device:', clf.device)

## 4. Entrenamiento

Configuración recomendada: `lr=2e-5`, `batch_size=16`, `epochs=3` (Sun et al., 2019).

In [ ]:
history = clf.fit(t_tr, a_tr, y_tr, epochs=3, batch_size=16, learning_rate=2e-5, seed=42)
history


## 5. Evaluación

In [ ]:
preds = clf.predict(t_te, a_te)
evaluate_predictions(y_te, preds)

## 6. Guardar modelo

In [ ]:
clf.save(REPO_ROOT / "models/beto")
